# 4. Readout Optimization

In [ ]:
n_average = 12

ro_amp_min = 0
ro_amp_max = 1
ro_amp_points = 11

ro_amp_arr = np.linspace(ro_amp_min, ro_amp_max, ro_amp_points)

In [ ]:
lsg_q0["measure_line"].range = -20

pulse_length = 2e-6

readout_weighting_function_sw = pulse_library.const(
            uid="readout_weighting_function_sw", length=pulse_length, amplitude=1.0
        )

In [ ]:
rabi_sw_res = []
popt_rabi_sw = []
pcov_rabi_sw = []

In [ ]:
for ro_amp in ro_amp_arr:
    print('Measure at amplitude:', round(ro_amp, 5))
    
    # #Set readout pulse
    # readout_pulse_sw = pulse_library.gaussian_square(
    #     uid="readout_pulse",
    #     length=pulse_length, #qubit_parameters["ro_len"],
    #     amplitude=round(ro_amp, 5),
    # )

    readout_pulse_sw = pulse_library.gaussian_square(
        uid="readout_pulse",
        length=qubit_parameters["ro_len"],
        amplitude=round(ro_amp, 5),
        width = qubit_parameters["ro_len"]*0.95,
    )
    
    #experiment definition
    exp_rabi_sw = make_rabi(rabi_sweep, 
                     gaussian_pulse, 
                     readout_pulse_sw, 
                     readout_weighting_function_sw, 
                     qubit_parameters["relax"], 
                     n_average)

    # set signal map for rabi experiment no experimental calibration necessary, 
    #calibration taken from DeviceSetup, i.e. baseline
    exp_rabi_sw.set_signal_map(qubit_meas_map)
    # compile
    exp_rabi_sw_comp = my_session.compile(exp_rabi_sw) 
    rabi_sw_results = my_session.run(exp_rabi_sw_comp)
    
    # get measurement data returned by the instruments
    rabi_res = rabi_sw_results.get_data("q0_rabi")
    rabi_amp = rabi_sw_results.get_axis("q0_rabi")[0]
    
    rabi_sw_res.append(rabi_res)
    
    I_zero = np.real(rabi_res)[0]
    Q_zero = np.imag(rabi_res)[0]
    
    norm_signal = np.sqrt((np.real(rabi_res) - I_zero)**2 + (np.imag(rabi_res) - Q_zero)**2)
    
    popt_norm, pcov_norm = fit_Rabi(rabi_amp, norm_signal, 10, 1, 1, 0.048, plot=True)
    
    popt_rabi_sw.append(popt_norm)
    pcov_rabi_sw.append(pcov_norm)
    
popt_rabi_arr = np.array(popt_rabi_sw)
pcov_rabi_arr = np.array(pcov_rabi_sw)

In [ ]:
#get error from covariance matrix
err = []

for pcov in pcov_rabi_sw:
    err.append(np.sqrt(np.diag(pcov)))
    
err_arr = np.array(err)

In [ ]:
fig, ax = plt.subplots(2, 2, sharex = True, figsize = (10, 8))

fig.suptitle('Relative Errors for Rabi measurements', fontsize=16)

fig.supxlabel("Readout amplitude (a.u.)")
fig.supylabel("Error (a.u.)")

ll = 1
ul = -1

# ax[0,0].plot(ro_amp_arr, err_arr[:,0], ".k", label = 'freq')
# ax[1,0].plot(ro_amp_arr, err_arr[:,1], ".k", label = 'phase')
# ax[0,1].plot(ro_amp_arr, err_arr[:,2], ".k", label = 'amp')
# ax[1,1].plot(ro_amp_arr, err_arr[:,3], ".k", label = 'off')

ax[0,0].plot(ro_amp_arr[ll:ul], err_arr[ll:ul,0]/popt_rabi_arr[ll:ul,0], ".k", label = 'freq')
ax[1,0].plot(ro_amp_arr[ll:ul], err_arr[ll:ul,1]/popt_rabi_arr[ll:ul,1], ".k", label = 'phase')
ax[0,1].plot(ro_amp_arr[ll:ul], -err_arr[ll:ul,2]/popt_rabi_arr[ll:ul,2], ".k", label = 'amp')
ax[1,1].plot(ro_amp_arr[ll:ul], err_arr[ll:ul,3]/popt_rabi_arr[ll:ul,3], ".k", label = 'off')

ax[0,0].set_title('Frequency')
ax[1,0].set_title('Phase')
ax[0,1].set_title('Amplitude')
ax[1,1].set_title('Offset')

#save figure
figname = 'Rabi_optimiz_20dBm_'
file_path = get_path_to_file(figname, '.png', sample_parameters)
plt.savefig(file_path, dpi=600, format='png', metadata=None,
        bbox_inches=None, pad_inches=0.1,
        facecolor='auto', edgecolor='auto',
        backend=None,
       )

## 4.1 Readout Pulse Settings

In [ ]:
#Set the parameters
qubit_parameters.update_parameter("ro_amp", 0.5)

qubit_parameters.update_parameter("ro_len", 2e-6)

In [ ]:
print('Readout length: ', qubit_parameters["ro_len"])
print('Readout amplitude: ', qubit_parameters["ro_amp"])
print('Readout Wfunc length: ', readout_weighting_function.length)

In [ ]:
print(readout_pulse)

In [ ]:
# qubit readout pulse - here simple constant pulse
# readout_pulse = pulse_library.const(
#     uid="readout_pulse",
#     length=qubit_parameters["ro_len"],
#     amplitude=qubit_parameters["ro_amp"],
# )

# qubit readout pulse - here with gaussian rise and fall
readout_pulse = pulse_library.gaussian_square(
    uid="readout_pulse_g",
    length=qubit_parameters["ro_len"],
    amplitude=qubit_parameters["ro_amp"],
    width = qubit_parameters["ro_len"]*0.95,
)

In [ ]:
# Set the weightning function
readout_weighting_function = pulse_library.const(
            uid="readout_weighting_function", length=qubit_parameters["ro_len"], amplitude=1.0
        )

In [ ]:
#Set the readout
readout_opt = {'readout_type': 'pulse',
                'readout_pulse': readout_pulse,
                'readout_weighting_function': readout_weighting_function,
                'relax_time': qubit_parameters["relax"],
                'measure_freq': qubit_parameters["ro_freq"],
                'acquire_freq': qubit_parameters["ro_freq"],
                'readout_range': -25,
                'readout_delay': qubit_parameters['ro_int_delay']
               }

readout_opt = QBaseParameters(sample=qsample_params,
                             name='readout_opt',
                             parameters=readout_opt)

## 4.2 Spectroscopy of Resonator in Different States

In [ ]:
# frequency range of spectroscopy scan - around expected centre frequency as defined in qubit parameters
spec_range = 2e6
# how many frequency points to measure
spec_num = 101

# define sweep parameters for two qubits
freq_sweep_ST = LinearSweepParameter(
    uid="res_freq",
    start=qubit_parameters["ro_freq"] - spec_range / 2,
    stop=qubit_parameters["ro_freq"] + spec_range / 2,
    count=spec_num,
)

# freq_sweep_ST = LinearSweepParameter(
#     uid="res_freq",
#     start=91e6,
#     stop=99e6,
#     count=spec_num,
# )

# take how many averages per point: 2^n_average
n_average = 12

In [ ]:
exp_spec_g = create_res_spec_gef(freq_sweep_ST, 
                                 x180, 
                                 None, 
                                 readout_low,#readout_low,
                                 n_average, 
                                 level = 0)

exp_spec_e = create_res_spec_gef(freq_sweep_ST, 
                                 x180, 
                                 None, 
                                 readout_low,#readout_low,
                                 n_average, 
                                 level = 1)

exp_spec_f = create_res_spec_gef(freq_sweep_ST, 
                                 x180, 
                                 x180_ef, 
                                 readout_low,#readout_low,
                                 n_average, 
                                 level = 2)

In [ ]:
exp_spec_g_comp = my_session.compile(exp_spec_g)

exp_spec_e_comp = my_session.compile(exp_spec_e)

exp_spec_f_comp = my_session.compile(exp_spec_f)

In [ ]:
print('Spectroscopy for qubit in g-state:')
spec_g_res = my_session.run(exp_spec_g_comp)

print('Spectroscopy for qubit in e-state:')
spec_e_res = my_session.run(exp_spec_e_comp)

print('Spectroscopy for qubit in f-state:')
spec_f_res = my_session.run(exp_spec_f_comp)

meas_ready()

In [ ]:
# get measurement data returned by the instruments
res_0_res = spec_g_res.get_data("q0_res_spec_e")
# define a frequency axis from the parameters
res_0_freq = lo_settings["ro_lo"] + spec_g_res.get_axis("q0_res_spec_e")[0]

# get measurement data returned by the instruments
res_1_res = spec_e_res.get_data("q0_res_spec_e")
# define a frequency axis from the parameters
res_1_freq = lo_settings["ro_lo"] + spec_e_res.get_axis("q0_res_spec_e")[0]


# # get measurement data returned by the instruments
res_2_res = spec_f_res.get_data("q0_res_spec_f")
# define a frequency axis from the parameters
res_2_freq = lo_settings["ro_lo"] + spec_f_res.get_axis("q0_res_spec_f")[0]

In [ ]:
fig, ax = plt.subplots(2, 1, sharex = True, figsize = (10, 8))

fig.suptitle('Spectroscopy of readout resonator for three qubit states', fontsize=16)
fig.supxlabel('Readout frequency, GHz')

ax[0].set_title('Amplitude')
ax[1].set_title('Phase')

ax[0].plot(res_0_freq*1e-9, np.abs(res_0_res), 'b', label = 'ground')
ax[0].plot(res_1_freq*1e-9, np.abs(res_1_res), 'r', label = 'first exited')
ax[0].plot(res_2_freq*1e-9, np.abs(res_2_res), 'g', label = 'second exited')
ax[0].axvline(x = (lo_settings["ro_lo"] + qubit_parameters["ro_freq"])*1e-9)
ax[0].legend()
ax[0].set_ylabel('Amplitude, a.u.')

ax[1].plot(res_0_freq*1e-9, np.unwrap(np.angle(res_0_res)), 'b', label = 'ground')
ax[1].plot(res_1_freq*1e-9, np.unwrap(np.angle(res_1_res)), 'r', label = 'first exited')
ax[1].plot(res_2_freq*1e-9, np.unwrap(np.angle(res_2_res)), 'g', label = 'second exited')
ax[1].axvline(x = (lo_settings["ro_lo"] + qubit_parameters["ro_freq"])*1e-9)
ax[1].legend()
ax[1].set_ylabel('Phase, a.u.')

#save figure
figname = 'Readout_spec_for_g_e'
file_path = get_path_to_file(figname, '.png', sample_parameters)
plt.savefig(file_path, dpi=600, format='png', metadata=None,
        bbox_inches=None, pad_inches=0.1,
        facecolor='auto', edgecolor='auto',
        backend=None,
       )

In [ ]:
plt.title('Spectroscopy of readout resonator for three qubit states')

plt.plot(res_0_res.real, res_0_res.imag, 'b', label = 'ground')
plt.plot(res_1_res.real, res_1_res.imag, 'r', label = 'first exited')
plt.plot(res_2_res.real, res_2_res.imag, 'g', label = 'second exited')

plt.xlabel('Real part, a.u.')
plt.ylabel('Imaginary part, a.u.')

plt.legend()

In [ ]:
plt.title('Spectroscopy of readout resonator for three qubit states: distance')

plt.plot(res_1_freq*1e-9, np.abs(res_0_res-res_1_res), 'r', label = '0-1')
plt.plot(res_1_freq*1e-9, np.abs(res_1_res-res_2_res), 'b', label = '1-2')
plt.plot(res_1_freq*1e-9, np.abs(res_0_res-res_2_res), 'g', label = '0-2')
plt.axvline(x = (lo_settings["ro_lo"] + qubit_parameters["ro_freq"])*1e-9)
plt.axvline(x = (lo_settings["ro_lo"] + qubit_parameters["ro_freq"])*1e-9, ls = '--', label = 'readout')


max_dist_arg = np.argmax(np.abs(res_0_res-res_1_res))
print('Optimat readout frequency:', res_1_freq[max_dist_arg]*1e-6, 'MHz')
plt.axvline(x = res_1_freq[max_dist_arg]*1e-9, ls = '-.', label = 'optimal')

max_1_arg = np.argmax(np.abs(res_1_res))
plt.axvline(x = res_1_freq[max_1_arg]*1e-9, ls = '-.', label = 'optimal')

# plt.axvline(x = 6.893, ls = ':', label = 'manual optimal')

plt.legend()

figname = 'Readout_spec_for_g_e_'
file_path = get_path_to_file(figname, '.png', sample_parameters)
plt.savefig(file_path, dpi=600, format='png', metadata=None,
        bbox_inches=None, pad_inches=0.1,
        facecolor='auto', edgecolor='auto',
        backend=None,
       )

In [ ]:
#Define optimal readout parameter
#preset optimal frequency for ge
try:
    qubit_parameters.update_parameter("ro_freq_opt", res_1_freq[max_dist_arg]-lo_settings["ro_lo"])
except Exception as ex:
    qubit_parameters["ro_freq_opt"] = res_1_freq[max_dist_arg]-lo_settings["ro_lo"]
# qubit_parameters.update_parameter("ro_freq_opt", 6.893e9 - lo_settings["ro_lo"])

print(f'ro freq opt: {(qubit_parameters["ro_freq_opt"] + lo_settings["ro_lo"]) * 1e-9:.6f} GHz')

In [ ]:
Data_gef_spec = {'ground_res': res_0_res,
                'ground_freq': res_0_freq,
                'fst_ex_res': res_1_res,
                'fst_ex_freq': res_1_freq,
                # 'scnd_ex_res': res_2_res,
                # 'scnd_ex_freq': res_2_freq,
            }

Data_gef_spec.update(qubit_parameters._params)
Data_gef_spec.update(lo_settings._params)

file_name = 'ge_spec_readout_test_'
file_path = get_path_to_file(file_name, '.mat', sample_parameters)
savemat(file_path, Data_gef_spec)

In [ ]:
qubit_parameters.update_parameter("relax", 120e-6)

In [ ]:
#define optimal readout

readout_opt = {'readout_type': 'pulse',
                'readout_pulse': readout_pulse,
                'readout_weighting_function': readout_weighting_function,
                'relax_time': qubit_parameters["relax"],
                'measure_freq': qubit_parameters["ro_freq_opt"],
                'acquire_freq': qubit_parameters["ro_freq_opt"],
                'readout_range': -20,
                'readout_delay': qubit_parameters['ro_int_delay']
               }